In [ ]:
https://stats.baseboll-softboll.se/api/v1/stats/events/2025-regionserien-baseboll/index?section=teams&stats-section=batting&team=&round=&split=&split=&language=en

In [1]:
import pandas as pd
import requests

url = "https://stats.baseboll-softboll.se/api/v1/stats/events/2025-regionserien-baseboll/index?section=teams&stats-section=batting&team=&round=&split=&split=&language=en"

headers_api = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'Accept': 'application/json'
}

try:
    response = requests.get(url, headers=headers_api)
    json_response = response.json()
    
    # --- 關鍵修正：進入 data 內部 ---
    # 使用 pd.json_normalize 處理嵌套結構
    df_stats = pd.json_normalize(json_response['data'], sep='_')
    
    # 打印出真正屬於球員數據的欄位清單
    print("內部數據的實際欄位：")
    print(df_stats.columns.tolist())
    
    # 根據 WBSC 系統 2025 年的命名規則，嘗試篩選正確欄位
    # 通常球員姓名在 'player_name' 或 'player_full_name'
    # 打擊率可能在 'avg' 或 'stats_avg'
    
    # 自動尋找包含關鍵字的欄位名稱
    def find_actual_col(keyword):
        matches = [c for c in df_stats.columns if keyword.lower() in c.lower()]
        return matches[0] if matches else None

    target_cols = {
        'player': find_actual_col('player'),
        'team': find_actual_col('team'),
        'avg': find_actual_col('avg'),
        'ops': find_actual_col('ops')
    }
    
    # 移除找不到的欄位
    final_cols = [v for v in target_cols.values() if v is not None]
    
    print("\n找到的對應欄位：", target_cols)
    print("\n數據預覽：")
    print(df_stats[final_cols].head())

except Exception as e:
    print(f"解析失敗: {e}")

內部數據的實際欄位：
['g', 'gs', 'ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'tb', 'avg', 'slg', 'obp', 'ops', 'bb', 'hbp', 'so', 'gdp', 'sf', 'sh', 'sb', 'cs', 'link', 'name', 'teamid']

找到的對應欄位： {'player': None, 'team': 'teamid', 'avg': 'avg', 'ops': 'ops'}

數據預覽：
   teamid  avg  ops
0   36162  342  882
1   36171  324  922
2   36165  362  932
3   36172  311  873
4   36170  363  979
